# File 1 — YC Dataset Base

Scrapes **all** Y Combinator companies from **2020 to now** into a dated dataset
(`yc_dataset_base_<YYYY-MM-DD>.parquet` + `.xlsx`). Re-run any time to refresh; the
logic lives in the `yc_scouter` package so this notebook stays thin.

**Output switch** (see the parameters cell): `download` (get the files when done),
`drive` (save to Google Drive), or `commit` (leave in `data/` — used by GitHub Actions).

In [ ]:
# --- parameters (papermill overrides these) ---
output = "download"      # "download" | "drive" | "commit"
out_dir = "data"          # where the dated files are written
date = None                # None -> today (YYYY-MM-DD)
source_json = None         # offline/CI: load records from a local JSON instead of fetching
use_cache = False          # dev only: reuse cache_path instead of hitting the network
cache_path = None
drive_folder = "Project YC Scouter"

In [ ]:
# --- ensure the package is importable (Colab installs it; repo/CI already has it) ---
try:
    import yc_scouter  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)
    import yc_scouter  # noqa: F401

In [ ]:
# --- build the Base dataset: fetch -> normalize -> enrich -> score -> dated export ---
from pathlib import Path
from yc_scouter import pipeline

df, paths = pipeline.build_base(
    source_json=source_json,
    use_cache=use_cache,
    cache_path=Path(cache_path) if cache_path else None,
    out_dir=Path(out_dir),
    date=date,
)
print(f"Base dataset: {len(df)} companies, {df['batch_year'].min()}-{df['batch_year'].max()}")
print("Wrote:", paths["parquet"].name, "and", paths["xlsx"].name)

In [ ]:
# --- deliver per the output switch ---
if output == "download":
    try:
        from google.colab import files
        for p in paths.values():
            files.download(str(p))
    except Exception as e:
        print("download skipped (not in Colab):", e)
elif output == "drive":
    try:
        import shutil
        from google.colab import drive
        drive.mount("/content/drive")
        dest = Path(f"/content/drive/MyDrive/{drive_folder}")
        dest.mkdir(parents=True, exist_ok=True)
        for p in paths.values():
            shutil.copy2(p, dest / p.name)
        print("Saved to Drive:", dest)
    except Exception as e:
        print("drive save skipped:", e)
else:  # commit
    print("Files ready in", out_dir, "(GitHub Actions will commit them).")